# **Testing**

In [ ]:
! pip install agent-framework --pre


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import asyncio
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

# **Working in 3 layers info extraction**

In [12]:
# --- Prompt Definitions ---

SOURCE_AGENT_PROMPT = """
You are a Source Discovery Agent.
Your goal is to find trustworthy web sources and EXISTING STUDY PLANS to help prepare for a specific job interview.
Target:
- Company: {company_name}
- Job Role: {job_role}

Your responsibility is to identify authoritative web entry points where we can find:
1. Company culture, values, tech stack, and engineering practices.
2. Role-specific expectations, required skills, and day-to-day responsibilities.
3. Interview experiences, question patterns, and specific "Study Guides" or "Roadmaps" created by others for this role.

Search Strategy:
- Prefer official company pages (Careers, Engineering Blogs).
- Prioritize professional networks (LinkedIn, Glassdoor, Blind).
- SEEK OUT community-created study plans/roadmaps on GitHub, Medium, Reddit, or Dev.to.
- Include technical preparation hubs (LeetCode, GeeksforGeeks) specific to [{company_name}].

Output contract (STRICT):
- Return ONLY a valid Python list of URLs strings.
- Example: ["https://careers.google.com", "https://github.com/jdoe/google-interview-roadmap", "https://leetcode.com/company/google"]
"""

In [13]:
# Define Research Target
company_name = "Google"
job_role = "Business Developer"

client = AzureOpenAIChatClient(
    credential=AzureCliCredential()
)

# -------- Agent 1: Source Discovery --------
source_agent = client.as_agent(
    instructions=SOURCE_AGENT_PROMPT.format(company_name=company_name, job_role=job_role),
    name="SourceDiscoveryAgent"
)

print(SOURCE_AGENT_PROMPT.format(company_name=company_name, job_role=job_role))
sources = await source_agent.run()
print("Discovered Sources:", sources.text)


You are a Source Discovery Agent.
Your goal is to find trustworthy web sources and EXISTING STUDY PLANS to help prepare for a specific job interview.
Target:
- Company: Google
- Job Role: Business Developer

Your responsibility is to identify authoritative web entry points where we can find:
1. Company culture, values, tech stack, and engineering practices.
2. Role-specific expectations, required skills, and day-to-day responsibilities.
3. Interview experiences, question patterns, and specific "Study Guides" or "Roadmaps" created by others for this role.

Search Strategy:
- Prefer official company pages (Careers, Engineering Blogs).
- Prioritize professional networks (LinkedIn, Glassdoor, Blind).
- SEEK OUT community-created study plans/roadmaps on GitHub, Medium, Reddit, or Dev.to.
- Include technical preparation hubs (LeetCode, GeeksforGeeks) specific to [Google].

Output contract (STRICT):
- Return ONLY a valid Python list of URLs strings.
- Example: ["https://careers.google.com", 

In [9]:
TOPIC_AGENT_PROMPT = """
You are a Research & Topic Extraction Agent.
Your Goal: Create a "Key Topics to Prepare" study plan based on the provided trusted sources.

Context:
- Company: {company_name}
- Job Role: {job_role}

Sources to Analyze:
{trusted_sources}

Task:
1. Simulate visiting and reading the provided sources.
2. Extract relevant skills, competencies, and company-specific values.
3. Synthesize this into a structured list of key topics.

Rules:
- Do NOT invent topics not supported by the sources or standard role expectations.
- Differentiate between:
  - "Technical": Hard skills, Tools, Domain Knowledge, Functional Competencies (e.g., Coding, Sales Strategy, Financial Modeling, CRM tools).
  - "Behavioral": Soft skills, Culture fit, Leadership Principles, Communication.

Output contract (STRICT JSON ONLY):
{{
  "technical_topics": [
    {{ "topic": "Name", "importance": "High/Medium", "reason": "Justification from sources" }}
  ],
  "behavioral_topics": [
    {{ "topic": "Name", "importance": "High/Medium", "reason": "Justification from sources" }}
  ]
}}
"""

In [10]:
# -------- Agent 2: Topic Extraction (Direct from Sources) --------
trusted_sources = sources.text

topic_agent = client.as_agent(
    instructions=TOPIC_AGENT_PROMPT.format(
        company_name=company_name, 
        job_role=job_role,
        trusted_sources=trusted_sources
    ),
    name="TopicExtractionAgent"
)

topic_result = await topic_agent.run(
    f"Analyze these sources and generate the key topics to prepare for {job_role} at {company_name}."
)

print("Topics to Cover:", topic_result.text)

Topics to Cover: ```json
{
  "technical_topics": [
    {
      "topic": "Strategic Partnership Development",
      "importance": "High",
      "reason": "Google career pages and LinkedIn job listings emphasize identifying, structuring, and negotiating partnerships with external organizations to drive product and business growth."
    },
    {
      "topic": "Market Analysis and Opportunity Assessment",
      "importance": "High",
      "reason": "Glassdoor interview insights and Google Careers descriptions highlight evaluating market trends and partner ecosystems to inform go-to-market strategies."
    },
    {
      "topic": "Financial Modeling and Business Case Development",
      "importance": "Medium",
      "reason": "Candidates are expected to quantify partnership ROI and understand revenue impact, often mentioned in Glassdoor and Reddit interview experiences."
    },
    {
      "topic": "Go-to-Market Strategy Planning",
      "importance": "High",
      "reason": "Google’s Busi

In [ ]:
ATOMIC_TOPIC_AGENT_PROMPT = """
You are a Syllabus Decomposition Agent.
Your Goal: Break down high-level study topics into a COMPREHENSIVE list of small, atomic, actionable study units.

Context:
- Company: {company_name}
- Job Role: {job_role}

Input Data:
The user will provide a list of "Technical" (Hard Skills) and "Behavioral" (Soft Skills) topics.

Task:
For each high-level topic, generate an EXHAUSTIVE list of atomic sub-concepts.
- Atomic means: A single concept that can be studied, practiced, or tested in isolation.
- Example (Tech): "System Design" -> ["Load Balancing", "Consistent Hashing", "CAP Theorem"].
- Example (Non-Tech): "Sales Strategy" -> ["Pipeline Management", "Needs Analysis", "Closing Techniques", "Objection Handling"].

Output contract (STRICT JSON ONLY):
{{
  "atomic_study_plan": [
    {{
      "parent_topic": "High Level Topic Name",
      "category": "Technical | Behavioral",
      "atomic_units": [
        "Unit 1",
        "Unit 2",
        "Unit 3"
      ]
    }}
  ]
}}
"""

In [ ]:
# -------- Agent 3: Atomic Decomposition --------
topics_json = topic_result.text

atomic_agent = client.as_agent(
    instructions=ATOMIC_TOPIC_AGENT_PROMPT.format(
        company_name=company_name,
        job_role=job_role
    ),
    name="AtomicDecompositionAgent"
)

atomic_result = await atomic_agent.run(
    f"Decompose the following topics into atomic study units:\n{topics_json}"
)

print("Atomic Study Plan:", atomic_result.text)

Atomic Study Plan: ```json
{
  "atomic_study_plan": [
    {
      "parent_topic": "Strategic Business Development & Partnership Management",
      "category": "Technical",
      "atomic_units": [
        "Identifying potential partners and alliance opportunities",
        "Evaluating partnership fit and strategic alignment",
        "Developing partnership proposals and value propositions",
        "Building win-win partnership structures",
        "Managing partnership lifecycle and performance metrics",
        "Understanding channel partnerships and resellers",
        "Contract negotiation principles for alliances",
        "Legal and compliance considerations in partnerships",
        "Stakeholder management within partnerships",
        "Strategic account planning for partner relationships"
      ]
    },
    {
      "parent_topic": "Market & Competitive Analysis",
      "category": "Technical",
      "atomic_units": [
        "Defining target markets and segmentation methods",
 

In [ ]:
STUDY_PLAN_AGENT_PROMPT = """
You are a Personal Study Scheduler Agent.
Your Goal: Create a detailed week-by-week study schedule using ALL provided atomic study units.

Context:
- Company: {company_name}
- Job Role: {job_role}
- Timeline: {weeks} weeks

Input Data:
The user will provide a comprehensive list of atomic study units (Technical and Behavioral).

Task:
1. Distribute ALL atomic study units logically across {weeks} weeks. DO NOT SKIP ANY TOPICS.
2. Ensure a balanced mix of Technical and Behavioral topics each week.
3. Structure the weeks to progress from Foundations -> Core Concepts -> Advanced -> MockPrep.

Output contract:
- Return a valid Markdown schedule.
- For each week, group related atomic units under their Parent Topic.
- Format:
  ### Week X: [Theme]
  #### [Parent Topic Name]
  - [Atomic Unit 1]
  - [Atomic Unit 2]
  - [Atomic Unit 3]
  ...
  #### [Another Parent Topic]
  ...
"""

In [ ]:
# -------- Agent 4: Study Plan Generation --------
atomic_plan_json = atomic_result.text
weeks_for_preparation = 4

plan_agent = client.as_agent(
    instructions=STUDY_PLAN_AGENT_PROMPT.format(
        company_name=company_name,
        job_role=job_role,
        weeks=weeks_for_preparation
    ),
    name="StudyPlanAgent"
)

study_plan_result = await plan_agent.run(
    f"Create a {weeks_for_preparation}-week study plan using these study units:\n{atomic_plan_json}"
)

print("Final Study Plan:", study_plan_result.text)

Final Study Plan: ```markdown
### Week 1: Foundations
#### Sales Strategy
- Sales Funnel Stages
- Pipeline Management Techniques
- Needs Analysis
- Value Proposition Development

#### Market Analysis
- Market Segmentation
- Competitive Analysis Techniques
- SWOT Analysis
- PESTEL Analysis

#### Communication Skills
- Effective Listening Techniques
- Clarity of Expression
- Non-Verbal Communication Skills

#### Adaptability
- Embracing Change Management
- Responding to Unexpected Challenges
- Continuous Learning Mindset

### Week 2: Core Concepts
#### Sales Strategy
- Closing Techniques
- Objection Handling
- Sales Forecasting Methods
- Customer Segmentation Strategies

#### Market Analysis
- Industry Trend Research
- Customer Needs Assessment
- Market Size Estimation
- Benchmarks and Key Performance Indicators

#### CRM Tools Proficiency
- Understanding CRM Software Basics
- Data Entry and Management in CRM
- Creating and Managing Customer Profiles

#### Collaboration and Teamwork
- Bu